In [2]:
import os
import sys, platform
import numpy as np
import pandas as pd
import matplotlib
import flopy
import matplotlib.pyplot as plt
import shutil
import matplotlib.colors as mcolors
from scipy.ndimage import gaussian_filter


In [ ]:
cwd = os.getcwd()
fig_dir = os.path.join(cwd, "../figures")
model_dirs = os.path.join(cwd, "../model")
# Check if the figures and model directories exist, if not create them
os.makedirs(fig_dir, exist_ok=True)
os.makedirs(model_dirs, exist_ok=True)

In [ ]:

print(f"Python      : {platform.python_version()}")
print(f"FloPy       : {flopy.__version__}")
print(f"NumPy       : {np.__version__}")
print(f"Pandas      : {pd.__version__}")
print(f"Matplotlib  : {matplotlib.__version__}")
print()
print("If any of these fail to import, install with:")
print("  pip install flopy numpy pandas matplotlib scipy xarray")

# Partie I — Fondements Théoriques

*Cette formation s'adresse en priorité à des étudiants et enseignants de mathématiques
et de physique. Avant de construire le moindre modèle numérique, il est essentiel de
comprendre que **MODFLOW ne fait rien de magique** : c'est un solveur numérique qui
résout, cellule par cellule et pas de temps par pas de temps, une équation aux dérivées
partielles bien connue — l'équation de diffusion appliquée à l'écoulement souterrain.
Maîtriser cette équation, et la physique qui la sous-tend, est ce qui distingue un
technicien qui fait tourner un logiciel d'un modélisateur capable d'en interpréter,
d'en critiquer et d'en améliorer les résultats.*

---
## 1. La loi de Darcy et l'équation d'écoulement souterrain

### 1.1 Un peu d'histoire et de physique

En 1856, l'ingénieur français Henry Darcy étudie l'écoulement de l'eau à travers des
colonnes de sable pour la ville de Dijon. Il établit expérimentalement que le débit
$Q$ traversant une colonne de section $A$ est :

- **proportionnel** à la différence de charge hydraulique $\Delta h$ entre l'entrée et
  la sortie,
- **inversement proportionnel** à la longueur $L$ du trajet parcouru,
- proportionnel à un coefficient propre au matériau, la **conductivité hydraulique**
  $K$ (dimension : longueur/temps, ex. m/jour).

$$
Q = -K A \frac{\Delta h}{L}
$$

Le signe négatif traduit un principe physique fondamental, identique à celui de la loi
de Fourier pour la chaleur ou de la loi de Fick pour la diffusion de masse : **l'eau
s'écoule des zones de charge hydraulique élevée vers les zones de charge plus faible**,
donc dans le sens opposé au gradient de charge.

### 1.2 Forme différentielle et généralisation en 3D

En faisant tendre $L \to 0$, on obtient le **débit spécifique** (ou vitesse de Darcy)
$q$ — un vecteur, homogène à une vitesse, mais qui n'est **pas** la vitesse réelle de
l'eau (voir §1.3) :

$$
q = -K \, \frac{dh}{dl}
\qquad \Longrightarrow \qquad
\vec{q} = -K \, \nabla h
= -\left( K_x \frac{\partial h}{\partial x},\; K_y \frac{\partial h}{\partial y},\; K_z \frac{\partial h}{\partial z} \right)
$$

Dans un milieu **anisotrope** — ce qui est la règle plutôt que l'exception en contexte
de socle fracturé, comme nous le verrons à l'Étape 4 — $K$ n'est plus un simple
scalaire mais un **tenseur** :

$$
\vec{q} = -\mathbf{K} \, \nabla h,
\qquad
\mathbf{K} =
\begin{pmatrix}
K_{xx} & K_{xy} & K_{xz} \\
K_{yx} & K_{yy} & K_{yz} \\
K_{zx} & K_{zy} & K_{zz}
\end{pmatrix}
$$

Dans ce notebook, nous simplifions en supposant les axes principaux d'anisotropie
alignés avec la grille (tenseur diagonal : $K_x$, $K_y$, $K_z$), une hypothèse standard
et raisonnable pour un premier modèle.

### 1.3 Vitesse de Darcy vs. vitesse linéaire réelle

Le débit spécifique $q$ traverse la *section totale* de la colonne, alors que l'eau ne
circule en réalité que dans les pores/fractures connectés (porosité effective $n_e$).
La **vitesse linéaire moyenne réelle** — celle qui contrôle par exemple le temps de
transfert d'un contaminant — est :

$$
v = \frac{q}{n_e}
$$

**Note du modélisateur.** Une porosité effective mal estimée ne change *rien* au champ
de charge $h$ calculé par MODFLOW (qui ne dépend que de $K$), mais change complètement
les temps de transit estimés. C'est une source d'erreur fréquente en pratique.

### 1.4 De Darcy à l'équation d'écoulement : le rôle de la conservation de la masse

La loi de Darcy décrit *comment* l'eau se déplace. Pour savoir *comment la charge $h$
évolue dans l'espace et le temps*, il faut lui combiner un second principe physique
fondamental : la **conservation de la masse** (équation de continuité). Pour un volume
élémentaire de milieu poreux, ce bilan s'écrit :

$$
\frac{\partial q_x}{\partial x} + \frac{\partial q_y}{\partial y} + \frac{\partial q_z}{\partial z}
+ Q_s = S_s \frac{\partial h}{\partial t}
$$

où $Q_s$ regroupe les sources et puits volumiques (recharge positive, pompage négatif,
etc.) et $S_s$ est l'**emmagasinement spécifique** (voir §1.6). En substituant $q_x,
q_y, q_z$ par leur expression donnée par la loi de Darcy, on obtient **l'équation
générale de l'écoulement souterrain en milieu poreux** — celle que MODFLOW 6 résout
numériquement, couche par couche, maille par maille :

$$
\boxed{
\frac{\partial}{\partial x}\!\left(K_x \frac{\partial h}{\partial x}\right)
+ \frac{\partial}{\partial y}\!\left(K_y \frac{\partial h}{\partial y}\right)
+ \frac{\partial}{\partial z}\!\left(K_z \frac{\partial h}{\partial z}\right)
+ R
= S_s \frac{\partial h}{\partial t}
}
$$

C'est une **équation aux dérivées partielles (EDP) parabolique**, structurellement
identique à l'équation de la chaleur de Fourier. C'est précisément pour cela que
l'intuition physique développée en cours de thermique (diffusion, régime permanent
vs transitoire, conditions aux limites) se transpose presque telle quelle à
l'hydrogéologie.

- **En régime permanent** (ce que nous construisons dans les prochaines étapes), le
  système est à l'équilibre : $\partial h/\partial t = 0$, et l'équation se réduit à
  une équation de Laplace/Poisson :

$$
\frac{\partial}{\partial x}\!\left(K_x \frac{\partial h}{\partial x}\right)
+ \frac{\partial}{\partial y}\!\left(K_y \frac{\partial h}{\partial y}\right)
+ \frac{\partial}{\partial z}\!\left(K_z \frac{\partial h}{\partial z}\right)
+ R = 0
$$

- **En régime transitoire** (Notebook 2), le terme $S_s \,\partial h/\partial t$ est
  conservé : la charge évolue dans le temps en réponse aux sollicitations saisonnières.

### 1.5 Pourquoi une méthode *numérique* ? — le rôle des mathématiques appliquées

Cette EDP n'admet une solution **analytique exacte** que dans des cas très simplifiés :
géométrie régulière, propriétés homogènes, conditions aux limites simples. C'est le cas
de la solution de **Thiem** (1906), qui décrit l'écoulement radial permanent vers un
puits unique dans un aquifère captif homogène et infini :

$$
h(r) = h_w + \frac{Q}{2 \pi T} \ln\!\left(\frac{r}{r_w}\right),
\qquad T = K b \ \text{(transmissivité, m}^2\text{/jour)}
$$

où $h_w$ est la charge au puits (rayon $r_w$) et $Q$ le débit pompé. Nous traçons cette
solution ci-dessous : c'est un excellent exemple de ce que les mathématiques
*peuvent* résoudre à la main.

Mais dès que l'on a — comme dans notre modèle — plusieurs couches, une conductivité
hydraulique hétérogène, une rivière, plusieurs puits en interaction, et un régime
transitoire saisonnier, l'EDP devient **impossible à résoudre analytiquement**. Il faut
alors la **discrétiser** : remplacer les dérivées continues par des différences finies
sur une grille de $150 \times 100 \times 3$ mailles, ce qui transforme l'EDP en un
**immense système d'équations algébriques linéaires (ou quasi-linéaires)** — plusieurs
dizaines de milliers d'inconnues $h_{i,j,k}$ — que MODFLOW 6 résout par itérations
(méthode de Newton-Raphson pour les non-linéarités liées à la nappe libre, puis
résolution du système linéaire à chaque itération). C'est exactement le type de
problème que l'algèbre linéaire numérique et l'analyse numérique — cœurs de vos
programmes de mathématiques et de physique — permettent de traiter à grande échelle.

**Modeler's Note.** Quand nous vérifierons plus loin la convergence du solveur
(`outer_dvclose`, `inner_dvclose`, nombre d'itérations), nous serons littéralement en
train de contrôler la convergence d'une méthode itérative de résolution de système
linéaire — un sujet d'analyse numérique standard, appliqué ici à un problème
d'ingénierie environnementale réel.

In [ ]:
# ---------------------------------------------------------------------------
# Illustration : la solution analytique de Thiem (écoulement radial permanent
# vers un puits unique, aquifère captif homogène). Purement pédagogique --
# aucune variable ici n'est réutilisée par le reste du notebook.
# ---------------------------------------------------------------------------
K_illustration = 5.0       # m/jour, conductivité hydraulique hypothétique
b_illustration = 20.0      # m, épaisseur de l'aquifère captif
T_illustration = K_illustration * b_illustration   # transmissivité, m^2/jour
Q_illustration = 800.0     # m^3/jour, débit pompé
r_w = 0.15                 # m, rayon du puits
h_w = 100.0                # m, charge imposée au puits

r = np.linspace(r_w, 500, 400)
h_r = h_w + (Q_illustration / (2 * np.pi * T_illustration)) * np.log(r / r_w)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(r, h_r, color="navy", lw=2)
ax.axhline(h_w, color="grey", ls=":", lw=1)
ax.set_xlabel("Distance radiale au puits, r (m)")
ax.set_ylabel("Charge hydraulique, h(r) (m)")
ax.set_title("Solution analytique de Thiem : cône de rabattement en régime permanent")
ax.annotate(f"T = K·b = {T_illustration:.0f} m²/jour\nQ = {Q_illustration:.0f} m³/jour",
            xy=(0.6, 0.15), xycoords="axes fraction", fontsize=9,
            bbox=dict(boxstyle="round", fc="white", ec="grey"))
plt.tight_layout()
plt.savefig("../figures/theory_thiem_solution.pdf", dpi=150)
plt.show()

print("Rabattement total à r = 1 m :", round(h_r[np.argmin(np.abs(r-1))] - h_w, 2), "m (négatif = rabattement)")
print("Rabattement total à r = 100 m :", round(h_r[np.argmin(np.abs(r-100))] - h_w, 2), "m")

> **Exercice.** À partir de l'équation de Thiem ci-dessus, calculez à la main (puis
> vérifiez numériquement) le rayon $r$ auquel le rabattement devient inférieur à 0,1 m.
> Que se passe-t-il si vous doublez $T$ ? Si vous doublez $Q$ ? Retrouvez-vous ce
> comportement dans le graphique ?

### 1.6 Emmagasinement : d'où vient l'eau qui répond à une sollicitation ?

L'équation de la §1.4 fait intervenir $S_s$, l'**emmagasinement spécifique** (1/m) :
le volume d'eau libéré par un aquifère **captif**, par unité de volume, pour une baisse
de charge unitaire. Il résulte de deux effets physiques combinés — la compressibilité
de l'eau et celle du squelette solide de l'aquifère :

$$
S_s = \rho\, g\, (\alpha + n\, \beta)
$$

avec $\rho$ la masse volumique de l'eau, $g$ l'accélération de la pesanteur, $\alpha$ la
compressibilité de la matrice solide, $n$ la porosité, et $\beta$ la compressibilité de
l'eau. Ces effets sont minuscules à l'échelle d'un grain de sable, d'où des valeurs de
$S_s$ typiquement comprises entre $10^{-5}$ et $10^{-4}$ m⁻¹.

Pour un aquifère **libre** (à surface libre, comme notre Couche 1), un mécanisme
beaucoup plus efficace s'ajoute : le **drainage gravitaire réel** de la porosité lors de
la baisse de la nappe. On définit alors la **porosité de drainage** (specific yield)
$S_y$, sans dimension, typiquement 100 à 10 000 fois plus grande que $S_s$ :

$$
S_y \sim 0.05 \text{ à } 0.25 \quad \text{(contre } S_s \sim 10^{-5}\text{–}10^{-4}\text{ m}^{-1}\text{)}
$$

C'est pourquoi, dans les prochaines étapes, la Couche 1 (libre) utilisera $S_y$ et les
Couches 2–3 (captives) utiliseront $S_s$ — cette distinction physique aura un impact
direct et visible sur la vitesse de réponse du système aux sollicitations saisonnières
que nous étudierons au Notebook 2.

> **Qu'avons-nous appris ?** La loi de Darcy et la conservation de la masse suffisent,
> à elles seules, à dériver l'équation qui gouverne tout le reste de ce notebook. Cette
> équation admet des solutions analytiques élégantes dans des cas simples (Thiem), mais
> exige des méthodes numériques dès que la géométrie ou les propriétés deviennent
> réalistes — ce qui est précisément le rôle de MODFLOW 6 et de FloPy dans les sections
> suivantes.

---
## 2. Changement climatique, bilan hydrique et système aquifère

Cette section établit, avec des équations simples mais rigoureuses, la chaîne de
causalité complète que nous simulerons plus loin dans ce notebook et dans le
Notebook 2 : **moins de pluie + plus de chaleur → moins de recharge et plus
d'évapotranspiration → plus de pompage requis pour l'irrigation → rabattement
accru de la nappe.**

### 2.1 Le bilan hydrique : d'où vient (ou ne vient pas) la recharge

À l'échelle d'un bassin versant, la conservation de la masse appliquée au cycle de
l'eau s'écrit :

$$
P = \mathrm{ETR} + R + R_o \pm \Delta S
$$

où $P$ est la précipitation, $\mathrm{ETR}$ l'évapotranspiration réelle, $R$ la
recharge qui atteint effectivement la nappe, $R_o$ le ruissellement de surface, et
$\Delta S$ la variation de stock d'eau dans le sol. **La recharge n'est donc jamais
égale à la pluie** : c'est un reliquat, après satisfaction de l'évapotranspiration, du
ruissellement et du déficit d'humidité du sol.

Dans ce notebook (Étape 8), nous représentons cette relation de façon simplifiée mais
réaliste par une **réponse à seuil** :

$$
R = \eta \, \max(0,\ P - P_{\text{seuil}})
$$

où $P_{\text{seuil}}$ (mm) représente le déficit d'humidité du sol et l'évaporation de
surface à combler avant toute infiltration profonde, et $\eta$ (efficacité de recharge,
sans dimension, $0<\eta<1$) la fraction du surplus qui atteint effectivement la nappe.

### 2.2 L'évapotranspiration : le lien direct avec la température

L'évapotranspiration potentielle ($\mathrm{ETP}$) — la quantité d'eau qu'*atmosphère et
végétation pourraient* évaporer/transpirer si l'eau ne manquait jamais — augmente avec
la température de l'air. La formulation de référence en hydrologie, l'équation de
**Penman-Monteith** (FAO-56), s'écrit :

$$
\mathrm{ETP} = \frac{0.408\, \Delta (R_n - G) + \gamma \dfrac{900}{T + 273}\, u_2 (e_s - e_a)}
{\Delta + \gamma (1 + 0.34\, u_2)}
$$

où $T$ est la température de l'air, $R_n$ le rayonnement net, $u_2$ la vitesse du vent,
$e_s - e_a$ le déficit de pression de vapeur, $\Delta$ la pente de la courbe de
pression de vapeur saturante, et $\gamma$ la constante psychrométrique. Cette formule
complète nécessite plus de variables météorologiques que nous n'en disposons ici — mais
le point essentiel, pour un public de mathématiciens et physiciens, est que
$\mathrm{ETP}$ est une **fonction croissante et non linéaire de $T$**, principalement à
travers le terme $e_s(T)$, qui suit une loi de Clausius-Clapeyron :

$$
e_s(T) = 0.6108 \exp\!\left(\frac{17.27\, T}{T + 237.3}\right) \quad \text{(kPa, T en °C)}
$$

**Conséquence directe pour le changement climatique :** une hausse de température,
même modeste, entraîne une hausse *non négligeable* de $\mathrm{ETP}$ à cause de la
croissance exponentielle de $e_s(T)$ — c'est la raison physique et mathématique pour
laquelle les scénarios climatiques prévoient systématiquement une augmentation de la
demande évaporative, indépendamment de toute variation des précipitations.

Dans ce notebook, nous utiliserons une formulation simplifiée : le taux d'EVT maximal
appliqué par MODFLOW est proportionnel à l'$\mathrm{ETP}$ via un coefficient cultural
$k_c$ :

$$
\mathrm{EVT}_{\max} = k_c \cdot \mathrm{ETP}
$$

### 2.3 De la sécheresse climatique à la demande de pompage

Considérons une superficie agricole $A$ (m²) à irriguer. Le **déficit hydrique net** que
l'irrigation doit compenser, par unité de temps, est :

$$
D(t) = \mathrm{ETP}(t) - P_{\text{eff}}(t)
$$

où $P_{\text{eff}}$ est la fraction de pluie directement utile aux cultures. Le débit de
pompage requis pour combler ce déficit sur la surface $A$, avec une efficacité
d'irrigation $\varepsilon$ (typiquement 0,6–0,8 pour l'irrigation de surface), est :

$$
Q_{\text{pompage}}(t) = \frac{A \cdot D(t)}{\varepsilon}
= \frac{A \left[\mathrm{ETP}(t) - P_{\text{eff}}(t)\right]}{\varepsilon}
$$

**C'est l'équation clé qui relie le changement climatique au pompage :** si le climat
se réchauffe ($\mathrm{ETP}\uparrow$) et s'assèche ($P_{\text{eff}}\downarrow$), alors
$D(t)$ augmente, et donc $Q_{\text{pompage}}(t)$ augmente **mécaniquement** — même sans
aucun changement de pratique agricole ou de superficie cultivée. C'est exactement ce que
nous représenterons au Notebook 2 en construisant des scénarios de sécheresse où
`recharge_scale` diminue, `evt_scale` augmente, **et** `pumping_scale` augmente
*simultanément* — trois symptômes d'une seule et même cause climatique.

### 2.4 Comment ces sollicitations combinées se propagent-elles jusqu'à la charge $h$ ?

Revenons à l'équation d'écoulement du §1.4. Le terme source $R$ (recharge moins EVT)
diminue, tandis que le terme puits $Q_s$ (pompage) augmente en valeur absolue. Dans le
régime **linéaire** (aquifère captif, propriétés constantes), l'équation d'écoulement
est linéaire en $h$, ce qui permet d'invoquer le **principe de superposition** : l'effet
combiné de plusieurs sollicitations simultanées est la somme des effets de chaque
sollicitation prise isolément :

$$
\Delta h_{\text{total}} \;\approx\; \Delta h_{\big(\text{recharge} \downarrow\big)} \;+\; \Delta h_{\big(\text{EVT} \uparrow\big)} \;+\; \Delta h_{\big(\text{pompage} \uparrow\big)}
$$

C'est un résultat mathématique puissant : il justifie, par exemple, la méthode des
images utilisée en hydrogéologie analytique (puits image pour représenter une limite à
charge imposée). **Mais notre modèle est majoritairement non linéaire** : la Couche 1
est libre (`icelltype=1`), ce qui signifie que sa transmissivité $T = K\,(h -
z_{\text{base}})$ dépend elle-même de $h$ — l'équation n'est alors plus strictement
linéaire, et la superposition n'est plus qu'une **approximation**. C'est précisément
pour cette raison que MODFLOW 6 emploie un schéma de linéarisation de
**Newton-Raphson** : à chaque itération, le problème non linéaire est remplacé par une
approximation linéaire locale, résolue, puis la solution est raffinée jusqu'à
convergence. Nous observerons concrètement cette non-linéarité dans les prochaines
sections lorsque nous testerons des scénarios de pompage accru.

### 2.5 Résumé de la chaîne causale

$$
\underbrace{P\downarrow,\ T\uparrow}_{\text{changement climatique}}
\;\Longrightarrow\;
\underbrace{R\downarrow,\ \mathrm{ETP}\uparrow}_{\text{bilan hydrique, §2.1–2.2}}
\;\Longrightarrow\;
\underbrace{Q_{\text{pompage}}\uparrow}_{\text{demande d'irrigation, §2.3}}
\;\Longrightarrow\;
\underbrace{h\downarrow}_{\text{équation d'écoulement, §1.4 et §2.4}}
$$

C'est cette chaîne complète, du climat jusqu'à la charge hydraulique simulée, que nous
allons maintenant construire numériquement, étape par étape, dans le reste de ce
notebook — puis pousser plus loin au Notebook 2 avec trois scénarios de sécheresse de
sévérité croissante.

In [ ]:
# ---------------------------------------------------------------------------
# Illustration : sensibilité de la pression de vapeur saturante e_s(T),
# et donc de l'ETP, a une hausse de temperature (loi de Clausius-Clapeyron,
# formulation FAO-56). Purement pédagogique.
# ---------------------------------------------------------------------------
T_range = np.linspace(15, 45, 100)   # °C
es = 0.6108 * np.exp((17.27 * T_range) / (T_range + 237.3))  # kPa

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(T_range, es, color="firebrick", lw=2)
axes[0].set_xlabel("Température de l'air, T (°C)")
axes[0].set_ylabel("Pression de vapeur saturante, e_s (kPa)")
axes[0].set_title("Loi de Clausius-Clapeyron : croissance non linéaire avec T")

# Sensibilité relative : de(es)/dT, pour montrer que l'effet s'accélère avec T
des_dT = np.gradient(es, T_range)
axes[1].plot(T_range, des_dT, color="darkorange", lw=2)
axes[1].set_xlabel("Température de l'air, T (°C)")
axes[1].set_ylabel("d(e_s)/dT (kPa/°C)")
axes[1].set_title("Sensibilité de l'ETP à un réchauffement -- plus forte à haute température")

plt.tight_layout()
plt.savefig("../figures/theory_clausius_clapeyron.pdf", dpi=150)
plt.show()

T1, T2 = 28.0, 30.0  # exemple : +2°C
es1 = 0.6108 * np.exp((17.27 * T1) / (T1 + 237.3))
es2 = 0.6108 * np.exp((17.27 * T2) / (T2 + 237.3))
print(f"e_s a {T1}°C : {es1:.3f} kPa")
print(f"e_s a {T2}°C : {es2:.3f} kPa")
print(f"Augmentation relative pour +2°C : {100*(es2-es1)/es1:.1f}% -- une hausse de temperature "
      "modeste se traduit par une hausse non negligeable du pouvoir evaporant de l'atmosphere.")

> **Exercice.** À partir du graphique ci-dessus, estimez l'augmentation relative de
> $e_s$ (et donc, approximativement, de l'ETP) pour un réchauffement de +2°C partant de
> 20°C, puis partant de 35°C. Le changement climatique affecte-t-il l'évapotranspiration
> de la même façon selon le climat de départ ? Que cela implique-t-il pour une région
> déjà chaude comme le sud du Burkina Faso ?

> **Qu'avons-nous appris ?** Nous avons établi, avec des équations vérifiables et non de
> simples affirmations qualitatives, que le changement climatique agit sur le système
> aquifère par (au moins) trois canaux distincts mais liés : une recharge réduite, une
> évapotranspiration accrue, et une demande de pompage accrue. Ces trois effets seront
> intégrés simultanément dans les scénarios de sécheresse du Notebook 2.

## 3. Modèle Conceptuel

In [ ]:

fig, ax = plt.subplots(figsize=(10, 5))
# coupe schématique simple
ax.fill_between([0, 10], [0, 0], [3, 2.2], color="#c9a26d", label="Couche 1 : altérée/alluviale (libre)")
ax.fill_between([0, 10], [3, 2.2], [-1.5, -1.8], color="#8a7f6b", alpha=0.85, label="Couche 2 : socle fracturé supérieur")
ax.fill_between([0, 10], [-1.5, -1.8], [-5, -5.2], color="#5c5648", alpha=0.9, label="Couche 3 : socle fracturé profond")
ax.plot([0, 10], [2.6, 1.6], "b--", lw=2, label="Niveau de la nappe")
ax.annotate("Pluie -> Recharge", xy=(2, 3.2), xytext=(2, 4.2),
            arrowprops=dict(arrowstyle="->", color="steelblue"), ha="center", color="steelblue")
ax.annotate("Pertes par ETP\n(nappe peu profonde)", xy=(6, 2.4), xytext=(6, 4.2),
            arrowprops=dict(arrowstyle="->", color="seagreen"), ha="center", color="seagreen")
ax.plot([5], [1.9], "v", color="navy", markersize=14)
ax.annotate("Rivière\n(RIV)", xy=(5, 1.9), xytext=(5, -0.6), ha="center", color="navy")
ax.plot([8], [2.4], "o", color="crimson", markersize=10)
ax.annotate("Puits de pompage\n(WEL)", xy=(8, 2.4), xytext=(8.6, 3.4), color="crimson")
ax.axvline(0, color="k", lw=3); ax.axvline(10, color="k", lw=3)
ax.text(-0.4, 1.0, "CHD\n(nord)", rotation=90, va="center")
ax.text(10.15, -1.0, "CHD\n(sud)", rotation=90, va="center")
ax.set_xlim(-1, 11); ax.set_ylim(-5.5, 4.8)
ax.set_xticks([]); ax.set_yticks([])
ax.set_title("Modèle conceptuel : système à 3 couches, altération/socle fracturé (schéma, hors échelle)")
ax.legend(loc="lower center", ncol=1, bbox_to_anchor=(0.18, 0.0), fontsize=9, framealpha=0.9)
plt.tight_layout()
plt.savefig("../figures/conceptual_model.pdf", dpi=150)
plt.show()

In [ ]:


# Reproductibilité : on fixe la graine aléatoire utilisée pour chaque champ synthétique
# de ce notebook
RNG_SEED = 42
np.random.seed(RNG_SEED)

# ---------------------------------------------------------------------------
# Structure de répertoires du projet (créée automatiquement)
# ---------------------------------------------------------------------------
PROJECT_ROOT = os.path.join(cwd, "..")
SUBDIRS = ["model/ss_basic", "model/ss_full", "model/transient_baseline",
           "model/transient_reduced_recharge", "model/transient_increased_pumping",
           "figures", "data", "output"]
for d in SUBDIRS:
    os.makedirs(os.path.join(PROJECT_ROOT, d), exist_ok=True)
# os.makedirs("figures", exist_ok=True)  # copies pratiques pour l'enregistrement des figures

print("Structure de projet créée dans :", os.path.abspath(PROJECT_ROOT))
for d in SUBDIRS:
    print(" -", os.path.join(PROJECT_ROOT, d))

In [ ]:


def find_mf6_executable():
    """Look for an mf6 executable on PATH or in a local ./bin folder."""
    candidates = [shutil.which("mf6"), "../bin/mf6", "bin/mf6",
                  os.path.expanduser("~/.local/bin/mf6")]
    for c in candidates:
        if c and os.path.isfile(c) and os.access(c, os.X_OK):
            return os.path.abspath(c)
    return None

MF6_EXE = find_mf6_executable()

if MF6_EXE is None:
    print("MODFLOW 6 executable not found.")
    print("Install it with FloPy's helper utility (downloads the official USGS build):")
    print()
    print("    import flopy")
    print('    flopy.utils.get_modflow("./bin", subset="mf6")')
    print()
    print("...or download it manually from:")
    print("    https://github.com/MODFLOW-ORG/executables/releases")
    print("and set MF6_EXE to the full path of the 'mf6' binary before continuing.")
else:
    print("Found MODFLOW 6 executable at:", MF6_EXE)

In [ ]:
# ---------------------------------------------------------------------------
# Dimensions de la grille (section 1 du cahier des charges)
# ---------------------------------------------------------------------------
nlay, nrow, ncol = 3, 150, 100
delr, delc = 100.0, 100.0   # m, largeur de colonne / hauteur de ligne
Lx, Ly = ncol * delr, nrow * delc  # étendue du domaine : 10 000 m (E-O) x 15 000 m (N-S)

print(f"Grille : {nlay} couches x {nrow} lignes x {ncol} colonnes")
print(f"Taille de maille : {delr} m x {delc} m")
print(f"Étendue du domaine : {Lx/1000:.1f} km (E-O) x {Ly/1000:.1f} km (N-S)")
print(f"Nombre total de mailles actives (max) : {nlay*nrow*ncol:,}")

### 4. Surface topographique et altitudes des couches (synthétiques)

Nous construisons une surface synthétique de sommet du modèle avec :
1. une pente régionale douce (plus élevée au nord, plus basse au sud — cohérente avec
   le gradient CHD nord-sud que nous appliquerons), plus
2. un relief lisse et spatialement corrélé (un champ aléatoire lissé par un filtre
   gaussien — pas une vraie topographie, mais qui se *comporte* comme une topographie :
   continue, sans ruptures brutales irréalistes), plus
3. une incision de vallée peu profonde et méandreuse qui deviendra le corridor de la
   rivière à l'étape 5.

Les épaisseurs de couches sont elles aussi construites comme des champs aléatoires
lisses, de sorte que la couche 1 mesure environ 15–21 m, la couche 2 environ 24–32 m,
et la couche 3 environ 33–43 m — des valeurs raisonnables et *hypothétiques* pour un
profil d'altération/socle fracturé.

In [ ]:
def smooth_field(seed, sigma, lo, hi, shape=(nrow, ncol)):
    """Un champ synthétique reproductible et spatialement corrélé, remis à l'échelle sur [lo, hi]."""
    rng = np.random.default_rng(seed)
    f = gaussian_filter(rng.normal(size=shape), sigma=sigma)
    f = (f - f.min()) / (f.max() - f.min())
    return lo + f * (hi - lo)

row_idx = np.arange(nrow).reshape(-1, 1)
col_idx = np.arange(ncol).reshape(1, -1)

# Pente régionale : 340 m au nord (ligne 0) -> 295 m au sud (ligne nrow-1)
regional_slope = 340.0 - (row_idx / (nrow - 1)) * 45.0
relief = smooth_field(seed=1, sigma=4, lo=-4, hi=4)
top = regional_slope + relief   # diffusé (broadcast) sur (nrow, ncol)

# Une vallée nord-sud légèrement méandreuse qui accueillera la rivière (étape 5)
river_col_center = 50 + 8 * np.sin(np.arange(nrow) / 25.0)
river_col = np.clip(np.round(river_col_center).astype(int), 5, ncol - 6)
for r in range(nrow):
    dist = np.abs(col_idx[0] - river_col[r])
    top[r, :] -= 6.0 * np.exp(-(dist**2) / (2 * 6.0**2))   # incision de la vallée

thick1 = 18 + smooth_field(seed=2, sigma=5, lo=-3, hi=3)
thick2 = 28 + smooth_field(seed=3, sigma=5, lo=-4, hi=4)
thick3 = 38 + smooth_field(seed=4, sigma=5, lo=-5, hi=5)

botm1 = top - thick1
botm2 = botm1 - thick2
botm3 = botm2 - thick3
botm = np.stack([botm1, botm2, botm3])

print(f"Plage d'altitude du sommet : {top.min():.1f} - {top.max():.1f} m")
print(f"Plage d'épaisseur, couche 1 : {thick1.min():.1f} - {thick1.max():.1f} m")
print(f"Plage d'épaisseur, couche 2 : {thick2.min():.1f} - {thick2.max():.1f} m")
print(f"Plage d'épaisseur, couche 3 : {thick3.min():.1f} - {thick3.max():.1f} m")
print(f"Plage d'épaisseur totale de l'aquifère : {(top-botm3).min():.1f} - {(top-botm3).max():.1f} m")



### 5. Carte : grille du modèle, altitudes des couches


In [ ]:
def plot_map(ax, data, title, cmap="viridis", cbar_label="", vmin=None, vmax=None,
             overlay_river=True):
    im = ax.imshow(data, extent=[0, Lx, 0, Ly], origin="upper", cmap=cmap,
                    vmin=vmin, vmax=vmax, aspect="equal")
    if overlay_river:
        river_x = river_col * delr + delr/2
        river_y = Ly - (np.arange(nrow) * delc + delc/2)
        ax.plot(river_x, river_y, color="deepskyblue", lw=1.5, label="Rivière (prévue, étape 5)")
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)")
    cb = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cb.set_label(cbar_label)
    return im

fig, axes = plt.subplots(1, 3, figsize=(16, 5.2))
plot_map(axes[0], top, "Altitude de la surface (sommet de la couche 1)", cmap="terrain",
         cbar_label="Altitude (m)")
plot_map(axes[1], top - botm[0], "Épaisseur de la couche 1", cmap="YlOrBr", cbar_label="m")
plot_map(axes[2], botm[2], "Base du modèle (fond de la couche 3)", cmap="cividis",
         cbar_label="Altitude (m)")
plt.tight_layout()
plt.savefig("../figures/grid_and_elevations.pdf", dpi=150)
plt.show()


---
---
## 6. Étape 2 — Modèle MODFLOW 6 de base en régime permanent


In [ ]:
# ---------------------------------------------------------------------------
# Limites à charge imposée (CHD)
# ---------------------------------------------------------------------------
# Les charges CHD suivent l'altitude *locale* de la surface (moins un léger décalage)
# plutôt qu'une valeur unique et fixe. Cela garde la charge de la limite physiquement
# cohérente avec le fond de la couche 1 partout le long du bord, même là où le relief
# local relève ou abaisse le sommet -- une valeur unique fixe se retrouverait sous le
# fond de l'aquifère dans certaines colonnes et ferait échouer le package.

In [ ]:

CHD_OFFSET = 2.0  # m below local land surface

def build_chd(head_shift=0.0, all_sides=False):
    """Construit les données de période de stress pour le package CHD.
    head_shift : décalage additif uniforme (m), utilisé plus tard pour d'autres scénarios.
    all_sides  : fixe aussi les bords ouest/est -- utilisé seulement pour ce tout premier
                 modèle, avant que la rivière ne donne au domaine un exutoire naturel.
    """
    data = []
    for lay in range(nlay):
        for c in range(ncol):
            data.append((lay, 0, c, top[0, c] - CHD_OFFSET + head_shift))
            data.append((lay, nrow - 1, c, top[nrow-1, c] - CHD_OFFSET + head_shift))
        if all_sides:
            for r in range(1, nrow - 1):   # on saute les coins, déjà définis ci-dessus
                data.append((lay, r, 0, top[r, 0] - CHD_OFFSET + head_shift))
                data.append((lay, r, ncol - 1, top[r, ncol-1] - CHD_OFFSET + head_shift))
    return data

chd_spd_ns_only   = build_chd(all_sides=False)   # utilisé à partir de l'étape 5 (la rivière draine le domaine)
chd_spd_all_sides = build_chd(all_sides=True)    # utilisé seulement pour ce tout premier modèle

print(f"Mailles CHD (nord/sud seulement) : {len(chd_spd_ns_only)}")
print(f"Mailles CHD (quatre côtés, modèle étape 2) : {len(chd_spd_all_sides)}")

In [ ]:
# ---------------------------------------------------------------------------
# Propriétés hydrauliques uniformes (étape 2 -- l'hétérogénéité vient plus tard, étape 4)
# ---------------------------------------------------------------------------
K_LAYER_MEANS = {
    "Kx": [3.0, 0.6, 0.15],    # m/jour : couche altérée, socle fracturé supérieur, socle fracturé profond
    "Kz_over_Kx": [0.1, 0.1, 0.1],
}
kx_uniform = np.stack([np.full((nrow, ncol), k) for k in K_LAYER_MEANS["Kx"]])
kz_uniform = np.stack([kx_uniform[i] * K_LAYER_MEANS["Kz_over_Kx"][i] for i in range(nlay)])
icelltype = [1, 0, 0]   # couche 1 libre/convertible, couches 2-3 captives

# Recharge : fraction conservatrice des pluies sahéliennes typiques (~950-1100 mm/an) --
# environ 4-8% d'efficacité de recharge dans ce tout premier modèle à recharge uniforme.
rch_uniform = 0.00010   # m/jour (~37 mm/an)

print("Kx uniforme par couche (m/jour) :", K_LAYER_MEANS["Kx"])
print("Recharge uniforme (m/jour) :", rch_uniform, f"=> {rch_uniform*365*1000:.0f} mm/an")

In [ ]:
def build_gwf_steady(sim_name, sim_ws, kx_arr, kz_arr, icelltype_arr, rch_arr,
                      chd_data, riv_spd=None, wel_spd=None, evt_args=None, strt=None):
    """Construit un modèle GWF MODFLOW 6 en régime permanent. Les packages optionnels
    (riv/wel/evt) ne sont ajoutés que si des données sont fournies : cette même fonction
    sert donc à toutes les étapes en régime permanent de ce notebook (étape 2 jusqu'au
    modèle complet de l'étape 8)."""
    sim = flopy.mf6.MFSimulation(sim_name=sim_name, sim_ws=sim_ws, exe_name=executable)
    flopy.mf6.ModflowTdis(sim, time_units="days", nper=1, perioddata=[(1.0, 1, 1.0)])
    flopy.mf6.ModflowIms(sim, complexity="MODERATE", outer_dvclose=1e-4,
                          inner_dvclose=1e-5, outer_maximum=200, inner_maximum=100)
    gwf = flopy.mf6.ModflowGwf(sim, modelname=sim_name, save_flows=True,
                                newtonoptions="NEWTON UNDER_RELAXATION")
    flopy.mf6.ModflowGwfdis(gwf, nlay=nlay, nrow=nrow, ncol=ncol,
                             delr=delr, delc=delc, top=top, botm=botm)
    if strt is None:
        strt = np.stack([top.copy()] * nlay)
    flopy.mf6.ModflowGwfic(gwf, strt=strt)
    flopy.mf6.ModflowGwfnpf(gwf, icelltype=icelltype_arr, k=kx_arr, k33=kz_arr,
                             save_specific_discharge=True)
    flopy.mf6.ModflowGwfrcha(gwf, recharge=rch_arr)
    flopy.mf6.ModflowGwfchd(gwf, stress_period_data=chd_data)
    if riv_spd is not None:
        flopy.mf6.ModflowGwfriv(gwf, stress_period_data=riv_spd, save_flows=True)
    if wel_spd is not None:
        flopy.mf6.ModflowGwfwel(gwf, stress_period_data=wel_spd, save_flows=True)
    if evt_args is not None:
        surface, rate, depth = evt_args
        flopy.mf6.ModflowGwfevta(gwf, surface=surface, rate=rate, depth=depth)
    flopy.mf6.ModflowGwfoc(gwf, budget_filerecord=f"{sim_name}.cbc",
                            head_filerecord=f"{sim_name}.hds",
                            saverecord=[("HEAD", "ALL"), ("BUDGET", "ALL")])
    return sim, gwf


def run_sim(sim, verbose_on_fail=True):
    """Écrit et exécute une simulation ; affiche la sortie du solveur en cas d'échec."""
    sim.write_simulation(silent=True)
    success, buff = sim.run_simulation(silent=True)
    if not success and verbose_on_fail:
        print("\n".join(buff[-40:]))
    return success

In [ ]:
sim_basic, gwf_basic = build_gwf_steady(
    "ss_basic", os.path.join(PROJECT_ROOT, "model/ss_basic"),
    kx_uniform, kz_uniform, icelltype, rch_uniform,
    chd_data=chd_spd_all_sides)

success_basic = run_sim(sim_basic)
print("Modèle de base (étape 2) convergé :", success_basic)

### Diagnostics du modèle : convergence, bilan de masse

In [ ]:
def check_model_convergence(sim, model_name):
    """Diagnostic réutilisable : confirme que la simulation a tourné et que le solveur a convergé."""
    listing = sim.name_file.filename if hasattr(sim, "name_file") else "mfsim.lst"
    print(f"--- Contrôle de convergence : {model_name} ---")
    success, buff = sim.run_simulation(silent=True)
    print("Simulation réussie :", success)
    return success

def check_water_budget(gwf, kstpkper=None, tolerance_pct=1.0):
    """Diagnostic réutilisable : rapporte la fermeture globale du bilan de masse pour un
    pas de temps. Utilise les mêmes totaux entrée/sortie que ceux rapportés par MODFLOW 6."""
    cbc = gwf.output.budget()
    kk = kstpkper if kstpkper is not None else cbc.get_kstpkper()[-1]
    names = [n.decode().strip() if isinstance(n, bytes) else n for n in cbc.get_unique_record_names()]
    total_in, total_out = 0.0, 0.0
    for name in names:
        if name in ("FLOW-JA-FACE", "DATA-SPDIS"):
            continue
        rec = cbc.get_data(kstpkper=kk, text=name)[0]
        q = rec["q"] if rec.dtype.names is not None else rec.ravel()
        total_in += q[q > 0].sum()
        total_out += q[q < 0].sum()
    discrepancy_pct = 100 * (total_in + total_out) / max(abs(total_in), 1e-9)
    ok = abs(discrepancy_pct) < tolerance_pct
    print(f"  Total ENTRÉES = {total_in:,.1f} m3/j | Total SORTIES = {total_out:,.1f} m3/j "
          f"| Écart = {discrepancy_pct:.3f}% | {'OK' if ok else 'À VÉRIFIER'}")
    return discrepancy_pct

_ = check_water_budget(gwf_basic)


### Examen des charges, des gradients et des directions d'écoulement

In [ ]:
heads_basic = gwf_basic.output.head().get_data()
print("Plage de charges, toutes couches confondues :", np.nanmin(heads_basic), "-", np.nanmax(heads_basic), "m")

for lay in range(nlay):
    print(f"  Couche {lay+1} : {heads_basic[lay].min():.1f} - {heads_basic[lay].max():.1f} m")

pct_above_surface = 100 * (heads_basic[0] > top).sum() / top.size
print(f"\nMailles de la couche 1 avec charge simulée AU-DESSUS de la surface : {pct_above_surface:.0f}%")

### Étape 3 — Visualiser et interpréter le modèle de base

In [ ]:
def plot_head_map(ax, heads_2d, title, wells_df=None, river_col_arr=None,
                   cmap="Blues", vmin=None, vmax=None):
    im = ax.imshow(heads_2d, extent=[0, Lx, 0, Ly], origin="upper", cmap=cmap,
                    vmin=vmin, vmax=vmax, aspect="equal")
    cs = ax.contour(np.flipud(heads_2d), extent=[0, Lx, 0, Ly], colors="k",
                     linewidths=0.6, levels=12)
    ax.clabel(cs, inline=True, fontsize=6, fmt="%.0f")
    if river_col_arr is not None:
        rx = river_col_arr * delr + delr/2
        ry = Ly - (np.arange(nrow) * delc + delc/2)
        ax.plot(rx, ry, color="deepskyblue", lw=1.5)
    if wells_df is not None:
        ax.scatter(wells_df.x, wells_df.y, marker="v", color="crimson", s=45,
                   edgecolor="k", linewidth=0.5, zorder=5)
    ax.set_title(title, fontsize=11); ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)")
    cb = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04); cb.set_label("Charge (m)")
    return im

fig, axes = plt.subplots(1, 3, figsize=(16, 5.2))
for lay, ax in enumerate(axes):
    plot_head_map(ax, heads_basic[lay], f"Modèle de base (étape 2) : charge, couche {lay+1}")
plt.tight_layout()
plt.savefig("../figures/stage2_heads.pdf", dpi=150)
plt.show()

In [ ]:
# Conductivité hydraulique et distribution de la recharge utilisées dans ce modèle de base
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
plot_map(axes[0], kx_uniform[0], "Kx couche 1 (uniforme, étape 2)", cmap="plasma",
         cbar_label="m/jour", overlay_river=False)
im = axes[1].imshow(np.full((nrow, ncol), rch_uniform*1000*365), extent=[0, Lx, 0, Ly],
                     origin="upper", cmap="YlGnBu")
axes[1].set_title("Recharge (uniforme, étape 2)"); axes[1].set_xlabel("x (m)"); axes[1].set_ylabel("y (m)")
plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04, label="mm/an")
plt.tight_layout(); plt.savefig("../figures/stage2_inputs.pdf", dpi=150); plt.show()

In [ ]:
# Direction d'écoulement (débit spécifique) et contrôle simple du gradient, couche 1
cbc_basic = gwf_basic.output.budget()
spdis = cbc_basic.get_data(text="DATA-SPDIS")[0]
qx, qy, qz = flopy.utils.postprocessing.get_specific_discharge(spdis, gwf_basic)

fig, ax = plt.subplots(figsize=(6.5, 8))
im = plot_head_map(ax, heads_basic[0], "Couche 1 : charges et directions d'écoulement", cmap="Blues")
skip = 6
Xc, Yc = np.meshgrid(np.arange(ncol)*delr+delr/2, Ly - (np.arange(nrow)*delc+delc/2))
ax.quiver(Xc[::skip, ::skip], Yc[::skip, ::skip],
          qx[0][::skip, ::skip], -qy[0][::skip, ::skip],
          color="white", scale=2, scale_units="xy", width=0.005)
plt.tight_layout(); plt.savefig("../figures/stage2_flow_directions.pdf", dpi=150); plt.show()




In [ ]:
def plot_cross_section(heads_stack, row=None, col=None, title="Coupe",
                        wells_df=None):
    fig, ax = plt.subplots(figsize=(10, 4.5))
    xs = flopy.plot.PlotCrossSection(model=gwf_basic.simulation.get_model(gwf_basic.name),
                                      line={"row": row} if row is not None else {"column": col})
    xs.plot_array(heads_stack, cmap="Blues", alpha=0.9)
    xs.plot_grid(linewidth=0.2, color="0.5")
    cs = xs.contour_array(heads_stack, colors="k", linewidths=0.6)
    ax.clabel(cs, inline=True, fontsize=7, fmt="%.0f")
    ax.set_title(title); ax.set_xlabel("Distance le long de la coupe (m)"); ax.set_ylabel("Altitude (m NGF)")
    plt.tight_layout()
    return fig, ax

# Coupe est-ouest au milieu du domaine (traverse la vallée)
fig, ax = plot_cross_section(heads_basic, row=75,
    title="Modèle de base (étape 2) : coupe E-O (ligne 75) -- notez le monticule dans la vallée")
plt.savefig("../figures/stage2_xsection_ew.pdf", dpi=150); plt.show()

---
## 7. Étape 4 — Propriétés hydrauliques spatialement variables


In [ ]:
k1 = smooth_field(seed=11, sigma=5, lo=0.6, hi=6.0)     # Couche 1 : altérée/alluviale, m/j
k2 = smooth_field(seed=12, sigma=6, lo=0.05, hi=1.2)    # Couche 2 : socle fracturé supérieur
k3 = smooth_field(seed=13, sigma=7, lo=0.01, hi=0.3)    # Couche 3 : socle fracturé profond

# Linéament de fracture synthétique NO-SE, K renforcé, dans la couche 2
lineament = np.zeros((nrow, ncol))
for r in range(nrow):
    dist = np.abs(col_idx[0] - (20 + 0.4*r))
    lineament[r, :] = np.exp(-(dist**2) / (2 * 3.0**2))
k2 = k2 + 3.0 * lineament

kx = np.stack([k1, k2, k3])
kz = np.stack([k1*0.1, k2*0.15, k3*0.1])   # couche 2 un peu plus connectée verticalement (fractures)

ss = np.array([1e-5, 5e-5, 3e-5])     # emmagasinement spécifique, 1/m
sy = np.array([0.15, 0.08, 0.05])     # porosité de drainage (seule la couche 1, libre, compte vraiment)

print("Plages de Kx par couche (m/jour) :")
for lay in range(nlay):
    print(f"  Couche {lay+1} : {kx[lay].min():.3f} - {kx[lay].max():.3f}  (médiane {np.median(kx[lay]):.3f})")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 6.5))
for lay in range(nlay):
    plot_map(axes[lay], kx[lay], f"Kx — couche {lay+1} (valeurs synthétiques de formation)",
              cmap="viridis", cbar_label="m/jour", overlay_river=False)
    axes[lay].imshow(np.ma.masked_where(kx[lay] < kx[lay].max()*0.6, kx[lay]),
                      extent=[0, Lx, 0, Ly], origin="upper", cmap="autumn", alpha=0.0)
plt.tight_layout()
plt.savefig("../figures/stage4_hydraulic_conductivity.pdf", dpi=150)
plt.show()

---
## 8. Étape 5 — La rivière (package RIV)


In [ ]:
river_stage = np.array([top[r, river_col[r]] - 6.0 for r in range(nrow)])
river_bed_elev = river_stage - 2.0
riverbed_K = 1.0          # m/jour, conductivité synthétique des sédiments du lit
riverbed_thick = 1.0      # m
riverbed_width = 10.0     # m (on suppose une rivière large de ~10 m dans une maille de 100 m)
riverbed_length = delc    # m, une maille de grille le long de la rivière
riv_conductance = riverbed_K * (riverbed_width * riverbed_length) / riverbed_thick  # m2/j

riv_spd = [(0, r, int(river_col[r]), float(river_stage[r]), float(riv_conductance),
            float(river_bed_elev[r])) for r in range(nrow)]

print(f"Mailles de rivière : {len(riv_spd)} (une par ligne du modèle)")
print(f"Plage de niveau de la rivière : {river_stage.min():.1f} - {river_stage.max():.1f} m NGF")
print(f"Conductance du lit de la rivière : {riv_conductance:.1f} m2/jour par maille")

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 8))
plot_map(ax, top, "Surface topographique et tracé de la rivière", cmap="terrain",
          cbar_label="Altitude (m NGF)")
plt.tight_layout()
plt.show()

---
## 9. Étape 6 — Puits de pompage (package WEL)


In [ ]:
rng = np.random.default_rng(7)
wells, tries = [], 0
while len(wells) < 10 and tries < 5000:
    tries += 1
    r = rng.integers(15, nrow - 15)
    c = rng.integers(8, ncol - 8)
    if abs(c - river_col[r]) < 3:      # on évite de placer un puits littéralement sur la rivière
        continue
    lay = int(rng.choice([0, 1, 2]))
    rate = -float(rng.uniform(1000, 1500))   # m3/jour, négatif = prélèvement
    wells.append([f"WEL-{len(wells)+1:02d}", lay, int(r), int(c), rate])

well_df = pd.DataFrame(wells, columns=["well_id", "layer", "row", "col", "q_m3d"])
well_df["x"] = well_df["col"] * delr + delr/2
well_df["y"] = Ly - (well_df["row"] * delc + delc/2)
well_df["dist_to_river_m"] = [abs(well_df.col[i] - river_col[well_df.row[i]]) * delr
                                for i in well_df.index]
well_df

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 8))
plot_map(ax, top, "Emplacement des puits par rapport à la rivière et à la topographie", cmap="terrain",
          cbar_label="Altitude (m)", overlay_river=True)
ax.scatter(well_df.x, well_df.y, marker="v", color="red", s=60, edgecolor="k", zorder=5)
for _, w in well_df.iterrows():
    ax.annotate(w.well_id, (w.x, w.y), fontsize=7, xytext=(4, 4), textcoords="offset points")
plt.tight_layout()
plt.savefig("../figures/stage6_wells.pdf", dpi=150)
plt.show()

---
## 10. Étape 7 — Évapotranspiration (package EVT)

In [ ]:
evt_surface = top.copy()
evt_extdp = 4.0          # m, profondeur d'extinction (valeur synthétique de formation)
evt_rate_uniform = 0.0022  # m/jour, taux max ~800 mm/an (utilisé seulement pour le modèle statique étapes 7/8)

evt_spd_static = [(0, r, c, float(evt_surface[r, c]), evt_rate_uniform, evt_extdp)
                   for r in range(nrow) for c in range(ncol)]

print(f"Mailles EVT : {len(evt_spd_static)} (couche 1 seulement -- l'ETP ne retire que l'eau souterraine peu profonde)")
print(f"Profondeur d'extinction : {evt_extdp} m | Taux d'ETP max : {evt_rate_uniform} m/jour "
      f"(~{evt_rate_uniform*365*1000:.0f} mm/an)")

In [ ]:
depth_to_water = top - heads_basic[0]
et_active_fraction = np.clip(1 - depth_to_water / evt_extdp, 0, 1)  # 1 = taux d'ETP plein, 0 = pas d'ETP

fig, axes = plt.subplots(1, 2, figsize=(13, 6.5))
plot_map(axes[0], depth_to_water, "Profondeur de la nappe (charges de l'étape 2)",
          cmap="Blues_r", cbar_label="Profondeur (m, négatif = au-dessus de la surface)", overlay_river=False)
plot_map(axes[1], et_active_fraction, "Fraction du taux d'ETP maximal active",
          cmap="YlGn", cbar_label="Fraction du taux d'ETP max (0-1)", overlay_river=False)
plt.tight_layout()
plt.savefig("../figures/stage7_evt_vulnerability.pdf", dpi=150)
plt.show()

---
## 11. Étape 8 — Recharge spatialement variable

In [ ]:
rch_spatial = smooth_field(seed=21, sigma=6, lo=0.00008, hi=0.00035)  # m/jour

total_recharge_m3d = (rch_spatial * delr * delc).sum()
total_recharge_m3yr = total_recharge_m3d * 365

print(f"Plage de recharge : {rch_spatial.min()*1000*365:.0f} - {rch_spatial.max()*1000*365:.0f} mm/an")
print(f"Recharge moyenne : {rch_spatial.mean()*1000*365:.0f} mm/an")
print(f"Recharge totale du domaine : {total_recharge_m3d:,.0f} m3/jour  "
      f"= {total_recharge_m3yr:,.0f} m3/an")

fig, ax = plt.subplots(figsize=(6.5, 8))
plot_map(ax, rch_spatial*1000*365, "Recharge spatialement variable (valeurs synthétiques de formation)",
          cmap="Blues", cbar_label="mm/an", overlay_river=False)
plt.tight_layout()
plt.savefig("../figures/stage8_recharge.pdf", dpi=150)
plt.show()

---
## 12. Tout assembler — Le modèle complet en régime permanent

In [ ]:
sim_full, gwf_full = build_gwf_steady(
    "ss_full", os.path.join(PROJECT_ROOT, "model/ss_full"),
    kx, kz, icelltype, rch_spatial,
    chd_data=chd_spd_ns_only,
    riv_spd={0: riv_spd},
    wel_spd={0: [((int(w.layer), int(w.row), int(w.col)), float(w.q_m3d))
                  for w in well_df.itertuples()]},
    evt_args=(evt_surface, evt_rate_uniform, evt_extdp),
    strt=heads_basic)

success_full = run_sim(sim_full)
print("Modèle complet en régime permanent convergé :", success_full)
_ = check_water_budget(gwf_full)

In [ ]:
heads_full = gwf_full.output.head().get_data()
pct_above = 100 * (heads_full[0] > top).sum() / top.size
print(f"Plage de charges : {np.nanmin(heads_full):.1f} - {np.nanmax(heads_full):.1f} m")
print(f"Mailles de la couche 1 au-dessus de la surface : {pct_above:.1f}%  (bien plus élevé dans le modèle de l'étape 2)")

In [ ]:
# À partir d'ici, nous utilisons un ensemble plus riche de fonctions graphiques (cartes
# de charge avec flèches d'écoulement, et coupes basées sur FloPy) qui superposent
# courbes isopièzes, puits, rivière et flèches de débit spécifique sur une même carte --
# plus utile maintenant que le modèle comporte plusieurs éléments en interaction.

def plot_head_map(heads, layer, ax=None, title=None, cmap="viridis",
                   show_contours=True, wells=None, show_river=False, vmin=None, vmax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 7))
    h = heads[layer]
    im = ax.imshow(h, extent=[0, Lx, 0, Ly], origin="upper", cmap=cmap, vmin=vmin, vmax=vmax)
    if show_contours:
        cs = ax.contour(np.linspace(0, Lx, ncol), np.linspace(Ly, 0, nrow), h,
                         colors="white", linewidths=0.6, levels=12)
        ax.clabel(cs, inline=True, fontsize=6, fmt="%.0f")
    if show_river:
        river_x = river_col * delr + delr/2
        river_y = Ly - (np.arange(nrow) * delc + delc/2)
        ax.plot(river_x, river_y, color="deepskyblue", lw=1.5, label="Rivière")
    if wells is not None:
        ax.scatter(wells.x, wells.y, marker="v", color="red", s=40, edgecolor="k",
                    linewidth=0.5, label="Puits", zorder=5)
    ax.set_title(title or f"Charge simulée, couche {layer+1}", fontsize=11)
    ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)")
    cb = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04); cb.set_label("Charge (m NGF)")
    if wells is not None or show_river:
        ax.legend(loc="upper right", fontsize=8)
    return ax

def flow_direction_arrows(gwf, layer, ax, step=8, scale=25, color="k"):
    # Superpose les flèches de direction d'écoulement à partir du débit spécifique (DATA-SPDIS)
    cbc = gwf.output.budget()
    spdis = cbc.get_data(text="DATA-SPDIS")[-1]
    qx, qy, qz = flopy.utils.postprocessing.get_specific_discharge(spdis, gwf)
    X = (np.arange(ncol) + 0.5) * delr
    Y = Ly - (np.arange(nrow) + 0.5) * delc
    ax.quiver(X[::step], Y[::step], qx[layer, ::step, ::step], qy[layer, ::step, ::step],
              color=color, scale=scale, width=0.003, alpha=0.85)
    return ax

def plot_cross_section(gwf, heads, row=None, col=None, kx_arr=None, title=None):
    # Coupe nord-sud (colonne fixée) ou est-ouest (ligne fixée) des charges et du K,
    # construite avec l'utilitaire PlotCrossSection de FloPy.
    fig, ax = plt.subplots(figsize=(11, 4.5))
    line = {"row": row} if row is not None else {"column": col}
    xsect = flopy.plot.PlotCrossSection(model=gwf, line=line)
    if kx_arr is not None:
        pc = xsect.plot_array(kx_arr, cmap="viridis", alpha=0.85,
                               norm=mcolors.LogNorm(vmin=max(kx_arr.min(), 1e-3), vmax=kx_arr.max()))
        plt.colorbar(pc, ax=ax, fraction=0.03, pad=0.02, label="Kx (m/jour, échelle log)")
    xsect.plot_grid(linewidth=0.2, color="grey")
    try:
        xsect.contour_array(heads, levels=15, colors="white", linewidths=0.7)
    except Exception:
        pass
    ax.set_title(title or ("Coupe N-S" if col is not None else "Coupe O-E"))
    ax.set_ylabel("Altitude (m NGF)")
    plt.tight_layout()
    return fig, ax

print("Fonctions graphiques mises à jour et prêtes : plot_head_map, flow_direction_arrows, plot_cross_section")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 6.5))
for lay in range(nlay):
    plot_head_map(heads_full, lay, ax=axes[lay], title=f"Modèle complet — charge, couche {lay+1}",
                  wells=well_df if lay in well_df.layer.values else None, show_river=(lay==0))
    flow_direction_arrows(gwf_full, lay, axes[lay])
plt.tight_layout()
plt.savefig("../figures/stage10_full_heads.pdf", dpi=150)
plt.show()

### Gains et pertes le long de la rivière

In [ ]:
cbc_full = gwf_full.output.budget()
riv_flow = cbc_full.get_data(text="RIV")[0]
# Les enregistrements du bilan RIV sont écrits dans le même ordre que riv_spd (une maille par ligne)
riv_q = np.array([rec["q"] for rec in riv_flow])
riv_df = pd.DataFrame({"row": np.arange(nrow), "y_m": Ly - (np.arange(nrow)*delc + delc/2),
                        "q_m3d": riv_q})
riv_df["reach_type"] = np.where(riv_df.q_m3d < 0, "Drainant (aquifère -> rivière)",
                                  "Infiltrant (rivière -> aquifère)")

fig, ax = plt.subplots(figsize=(10, 4.5))
colors = np.where(riv_df.q_m3d < 0, "steelblue", "sienna")
ax.bar(riv_df.y_m, riv_df.q_m3d, width=90, color=colors)
ax.axhline(0, color="k", lw=0.8)
ax.set_xlabel("Position le long de la rivière, nord (0) au sud (15 000 m)")
ax.set_ylabel("Flux de la maille de rivière (m3/jour)\n[négatif = drainant, positif = infiltrant]")
ax.set_title("Gains et pertes le long de la rivière (régime permanent, modèle complet)")
plt.tight_layout()
plt.savefig("../figures/stage10_river_gains_losses.pdf", dpi=150)
plt.show()

print(riv_df.reach_type.value_counts())
print(f"Échange net total avec la rivière : {riv_df.q_m3d.sum():,.0f} m3/jour "
      f"({'gain net pour la rivière' if riv_df.q_m3d.sum()<0 else 'perte nette de la rivière'})")

In [ ]:
mid_row = nrow // 2
mid_col = ncol // 2
plot_cross_section(gwf_full, heads_full, col=mid_col, kx_arr=kx,
                    title=f"Coupe nord-sud (colonne {mid_col})")
plt.savefig("../figures/stage10_xsection_ns.pdf", dpi=150)
plt.show()

plot_cross_section(gwf_full, heads_full, row=mid_row, kx_arr=kx,
                    title=f"Coupe ouest-est (ligne {mid_row})")
plt.savefig("../figures/stage10_xsection_we.pdf", dpi=150)
plt.show()